# Feature Scaling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/01-data-preprocessing/03_feature_scaling.ipynb)


---

## What are we learning?

Feature scaling rescales numeric columns so they share a common range (often 0-1 or mean 0, std 1). This prevents variables with large magnitudes (like income) from dominating distance-based algorithms such as k-NN, PCA, or gradient descent.

## The idea in plain English

Imagine comparing apples to oranges when one is weighed in grams and the other in kilograms. Scaling puts every fruit on the same “gram scale” so the comparison is fair. Likewise, we rescale features so no single column shouts louder just because its numbers are bigger.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sns.set_theme(style='whitegrid')
print('Setup done!')

## Step 1 — Load data

In [ ]:
# Toy dataset: two features with very different scales
X, y = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=42)
# Artificially inflate one feature to mimic real-world scale mismatch
X[:, 0] = X[:, 0] * 1000 + 50000  # salary-like
X[:, 1] = X[:, 1] * 5 + 25        # age-like

df = pd.DataFrame(X, columns=['salary', 'age'])
df['label'] = y
df.head()

## Step 2 — Apply Feature Scaling

In [ ]:
# Min-Max scaling (0-1 range)
mm_scaler = MinMaxScaler()
df_minmax = df.copy()
df_minmax[['salary', 'age']] = mm_scaler.fit_transform(df[['salary', 'age']])

# Standard scaling (mean 0, std 1)
std_scaler = StandardScaler()
df_std = df.copy()
df_std[['salary', 'age']] = std_scaler.fit_transform(df[['salary', 'age']])

print('Original ranges:')
print(df[['salary', 'age']].agg(['min', 'max']))
print('\nMin-Max scaled ranges:')
print(df_minmax[['salary', 'age']].agg(['min', 'max']))
print('\nStandard scaled means & stds:')
print(df_std[['salary', 'age']].agg(['mean', 'std']).round(3))

## Step 3 — Visualise

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

# Original
sns.scatterplot(data=df, x='salary', y='age', hue='label', ax=axes[0], palette='Set2')
axes[0].set_title('Original')

# Min-Max scaled
sns.scatterplot(data=df_minmax, x='salary', y='age', hue='label', ax=axes[1], palette='Set2')
axes[1].set_title('Min-Max Scaled (0-1)')

# Standard scaled
sns.scatterplot(data=df_std, x='salary', y='age', hue='label', ax=axes[2], palette='Set2')
axes[2].set_title('Standard Scaled (z-score)')

plt.tight_layout()
plt.show()

## Results & interpretation

In [ ]:
# Compute Euclidean distances between first two points
def dist(a, b):
    return np.sqrt(np.sum((a - b)**2))

p0_orig = df.iloc[0][['salary', 'age']].values
p1_orig = df.iloc[1][['salary', 'age']].values

p0_mm   = df_minmax.iloc[0][['salary', 'age']].values
p1_mm   = df_minmax.iloc[1][['salary', 'age']].values

p0_std  = df_std.iloc[0][['salary', 'age']].values
p1_std  = df_std.iloc[1][['salary', 'age']].values

print('Distance between first two rows:')
print(f'Original:   {dist(p0_orig, p1_orig):.2f}')
print(f'Min-Max:    {dist(p0_mm, p1_mm):.4f}')
print(f'Standard:   {dist(p0_std, p1_std):.4f}')

print('\nNotice how the unscaled distance is dominated by salary.')

## Summary

- Feature scaling ensures all numeric columns contribute equally to distance-based models.
- Min-Max scaling squeezes values into a fixed 0-1 range; useful when bounds are known.
- Standard scaling centers data at 0 with unit variance; robust to outliers.
- Always fit the scaler on training data only, then transform train & test to avoid data leakage.
- Scaling is essential for k-NN, PCA, k-means, SVM, and neural networks.

## Exercises

1. Apply RobustScaler to the same dataset and compare the resulting ranges.
2. Build a k-NN classifier (k=3) on both original and scaled data; report accuracy differences.
3. Create a Pipeline that chains StandardScaler and logistic regression, then evaluate with cross_val_score.

In [ ]:
# Your code here